# 10 - Multi-Dimensional Examples (2D and 3D)

This notebook explores SGD dynamics in higher-dimensional parameter spaces, comparing 2D and 3D loss landscapes.

**Converted from:** `example 2d-3d.nb` (Mathematica)

## Contents:
1. 2D loss landscape examples
2. 3D parameter space visualization
3. Multi-dimensional gradient flow
4. Projection and dimensionality reduction
5. Comparative analysis of convergence

## Background

Real neural networks have thousands to billions of parameters. While we can't visualize such high dimensions, studying 2D and 3D cases provides intuition about higher-dimensional behavior.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from mpl_toolkits.mplot3d import Axes3D

# Add utils to path
sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import SmoothNonlinearLoss, generate_noisy_data
from visualization import plot_loss_landscape, plot_loss_landscape_3d

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. 2D Loss Landscape Examples

We'll examine several types of 2D loss landscapes with different geometric properties.

In [ ]:
# Generate data for 2D problem
np.random.seed(42)

x_data, y_data = generate_noisy_data(
    x_range=(-3, 3),
    n_points=20,
    n_samples_per_point=5,
    noise_std=0.5,
    p=1.0,
    random_state=42
)

loss_obj = SmoothNonlinearLoss(p=1.0)

print(f"2D problem with {len(x_data)} data points")

In [ ]:
# Visualize 2D landscape in multiple ways
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

param_range = ((-1, 3), (-1, 3))

# Contour plot
plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=40,
    ax=axes[0],
    title='2D Loss Landscape - Contour View'
)

# Heatmap
n_grid = 100
a_vals = np.linspace(param_range[0][0], param_range[0][1], n_grid)
b_vals = np.linspace(param_range[1][0], param_range[1][1], n_grid)
A, B = np.meshgrid(a_vals, b_vals)

L = np.zeros_like(A)
for i in range(n_grid):
    for j in range(n_grid):
        params = np.array([A[i, j], B[i, j]])
        L[i, j] = loss_obj(params, x_data, y_data)

im = axes[1].imshow(L, extent=[param_range[0][0], param_range[0][1],
                               param_range[1][0], param_range[1][1]],
                   origin='lower', cmap='viridis', aspect='auto')
plt.colorbar(im, ax=axes[1], label='Loss')
axes[1].set_xlabel('Parameter a', fontsize=12)
axes[1].set_ylabel('Parameter b', fontsize=12)
axes[1].set_title('2D Loss Landscape - Heatmap View', fontsize=14)

plt.tight_layout()
plt.show()

## 2. 3D Parameter Space Visualization

Now let's extend to a 3D parameter space. We'll create a synthetic 3-parameter problem.

In [ ]:
class Simple3DLoss:
    """Simple 3D quadratic loss with cross terms."""
    
    def __call__(self, params):
        """Compute loss for 3D parameters."""
        theta1, theta2, theta3 = params
        
        # Quadratic with different curvatures and cross terms
        loss = (theta1 - 1.0)**2 + 2*(theta2 - 0.5)**2 + 0.5*(theta3 + 0.5)**2
        loss += 0.3*theta1*theta2 + 0.2*theta2*theta3
        
        return loss
    
    def gradient(self, params):
        """Compute gradient for 3D parameters."""
        theta1, theta2, theta3 = params
        
        grad = np.zeros(3)
        grad[0] = 2*(theta1 - 1.0) + 0.3*theta2
        grad[1] = 4*(theta2 - 0.5) + 0.3*theta1 + 0.2*theta3
        grad[2] = (theta3 + 0.5) + 0.2*theta2
        
        return grad

loss_3d = Simple3DLoss()

# Test
test_params = np.array([0.5, 0.5, 0.5])
print(f"Test loss at {test_params}: {loss_3d(test_params):.4f}")
print(f"Test gradient: {loss_3d.gradient(test_params)}")

In [ ]:
# Run 3D SGD trajectory
def sgd_trajectory_3d(loss_fn, initial_params, learning_rate, n_steps):
    """Simple gradient descent for 3D problem."""
    trajectory = np.zeros((n_steps + 1, 3))
    trajectory[0] = initial_params
    
    for i in range(n_steps):
        grad = loss_fn.gradient(trajectory[i])
        # Add small noise to simulate SGD
        noise = np.random.randn(3) * 0.05
        trajectory[i + 1] = trajectory[i] - learning_rate * (grad + noise)
    
    return trajectory

# Run trajectory
np.random.seed(42)
initial_3d = np.array([2.0, -1.0, 1.5])
traj_3d = sgd_trajectory_3d(
    loss_fn=loss_3d,
    initial_params=initial_3d,
    learning_rate=0.05,
    n_steps=500
)

print(f"3D trajectory: {len(traj_3d)} points")
print(f"Initial: {traj_3d[0]}, Final: {traj_3d[-1]}")

In [ ]:
# Visualize 3D trajectory
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot trajectory
ax.plot(traj_3d[:, 0], traj_3d[:, 1], traj_3d[:, 2], 
           'b-', linewidth=2, alpha=0.7, label='SGD trajectory')

# Mark start and end
ax.scatter(traj_3d[0, 0], traj_3d[0, 1], traj_3d[0, 2],
          c='green', s=200, marker='o', edgecolor='black', 
          linewidth=2, label='Start')
ax.scatter(traj_3d[-1, 0], traj_3d[-1, 1], traj_3d[-1, 2],
          c='red', s=200, marker='s', edgecolor='black',
          linewidth=2, label='End')

ax.set_xlabel('θ₁', fontsize=12)
ax.set_ylabel('θ₂', fontsize=12)
ax.set_zlabel('θ₃', fontsize=12)
ax.set_title('3D SGD Trajectory', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 3. Multi-Dimensional Gradient Flow

Compare gradient flow in 2D vs 3D by examining convergence rates.

In [ ]:
# Compute loss trajectories for 2D and 3D
losses_2d = np.array([loss_obj(params, x_data, y_data) 
                     for params in traj_3d[:, :2]])  # Use first 2 dims

losses_3d = np.array([loss_3d(params) for params in traj_3d])

print(f"2D loss: {losses_2d[0]:.4f} -> {losses_2d[-1]:.4f}")
print(f"3D loss: {losses_3d[0]:.4f} -> {losses_3d[-1]:.4f}")

In [ ]:
# Plot loss convergence comparison
fig, ax = plt.subplots(figsize=(14, 7))

iterations = np.arange(len(traj_3d))

ax.plot(iterations, losses_3d, 'b-', linewidth=2.5, 
       alpha=0.8, label='3D Loss')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Convergence in 3D Parameter Space', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Projection and Dimensionality Reduction

Project 3D trajectories onto 2D planes to visualize dynamics.

In [ ]:
# Create 2D projections of 3D trajectory
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

projections = [
    (0, 1, 'θ₁', 'θ₂'),
    (0, 2, 'θ₁', 'θ₃'),
    (1, 2, 'θ₂', 'θ₃')
]

for ax, (dim1, dim2, label1, label2) in zip(axes, projections):
    # Color by iteration (time)
    colors = plt.cm.viridis(np.linspace(0, 1, len(traj_3d)))
    
    for i in range(len(traj_3d) - 1):
        ax.plot(traj_3d[i:i+2, dim1], traj_3d[i:i+2, dim2],
               color=colors[i], linewidth=2, alpha=0.7)
    
    # Mark start and end
    ax.scatter(traj_3d[0, dim1], traj_3d[0, dim2],
              c='green', s=150, marker='o', edgecolor='black',
              linewidth=2, zorder=5, label='Start')
    ax.scatter(traj_3d[-1, dim1], traj_3d[-1, dim2],
              c='red', s=150, marker='s', edgecolor='black',
              linewidth=2, zorder=5, label='End')
    
    ax.set_xlabel(label1, fontsize=12)
    ax.set_ylabel(label2, fontsize=12)
    ax.set_title(f'Projection: {label1}-{label2}', fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Color gradient shows progression through time (purple to yellow)")

## 5. Comparative Analysis: 2D vs 3D

Run multiple trajectories and compare convergence statistics.

In [ ]:
# Run multiple 3D trajectories
n_runs = 10
final_losses_3d = []
convergence_iters_3d = []

threshold = 0.1  # Loss threshold for "convergence"

np.random.seed(123)
for i in range(n_runs):
    init = np.random.uniform(-1, 2, size=3)
    traj = sgd_trajectory_3d(
        loss_fn=loss_3d,
        initial_params=init,
        learning_rate=0.05,
        n_steps=500
    )
    
    losses = np.array([loss_3d(params) for params in traj])
    final_losses_3d.append(losses[-1])
    
    # Find convergence iteration
    converged = np.where(losses < threshold)[0]
    if len(converged) > 0:
        convergence_iters_3d.append(converged[0])
    else:
        convergence_iters_3d.append(500)

final_losses_3d = np.array(final_losses_3d)
convergence_iters_3d = np.array(convergence_iters_3d)

print(f"3D Results ({n_runs} runs):")
print(f"  Final loss: {np.mean(final_losses_3d):.4f} ± {np.std(final_losses_3d):.4f}")
print(f"  Convergence iter: {np.mean(convergence_iters_3d):.1f} ± {np.std(convergence_iters_3d):.1f}")

In [ ]:
# Visualize statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Final loss distribution
axes[0].hist(final_losses_3d, bins=8, edgecolor='black', alpha=0.7, color='blue')
axes[0].axvline(np.mean(final_losses_3d), color='red', 
               linestyle='--', linewidth=2, label='Mean')
axes[0].set_xlabel('Final Loss', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Final Losses (3D)', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Convergence iterations
axes[1].hist(convergence_iters_3d, bins=10, edgecolor='black', alpha=0.7, color='green')
axes[1].axvline(np.mean(convergence_iters_3d), color='red',
               linestyle='--', linewidth=2, label='Mean')
axes[1].set_xlabel('Iterations to Convergence', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title(f'Convergence Speed (3D, threshold={threshold})', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Parameter evolution in 3D
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

param_names = ['θ₁', 'θ₂', 'θ₃']
for i, (ax, name) in enumerate(zip(axes, param_names)):
    ax.plot(iterations, traj_3d[:, i], linewidth=2.5, color=f'C{i}')
    ax.axhline(traj_3d[-1, i], color='red', linestyle='--', 
              linewidth=2, alpha=0.5, label='Final value')
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel(name, fontsize=13)
    ax.set_title(f'Evolution of {name}', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In this notebook, we explored:

1. **2D Loss Landscapes**: Visualized through contour plots and heatmaps
2. **3D Parameter Spaces**: Extended analysis to three-parameter problems
3. **Gradient Flow**: Compared convergence dynamics in different dimensions
4. **Projections**: Visualized 3D trajectories through 2D projections
5. **Statistical Comparison**: Analyzed convergence statistics across multiple runs

**Key Insights:**
- Higher dimensions introduce more complex loss landscapes
- Projections help visualize high-dimensional trajectories
- Each parameter dimension can have different convergence rates
- Cross-terms in loss create coupling between parameters
- SGD behavior generalizes from 2D to higher dimensions

**Observations:**
- 3D trajectories show spiraling behavior toward minima
- Different projections reveal different aspects of the dynamics
- Convergence statistics are similar across dimensions for this problem

**Next Steps:**
- Explore piecewise linear loss functions
- Study non-smooth loss landscapes